# FOMO — SJSU Headcount Training & Visualization

End-to-end notebook that:
1. Clones [fomo-edge-ai/fomo](https://github.com/fomo-edge-ai/fomo) and installs it
2. Downloads the **SJSU Headcount Scene-1** dataset from HuggingFace (`bdanko/sjsu-headcount-scene-1`)
3. Trains **FOMO-m** (the lightweight MobileNetV2 point-localization model) for 10 epochs
4. Visualizes the predictions on validation images — predicted person locations as coloured circles overlaid on the original image

## 1 — Environment Setup

In [1]:
!pip install fomo-edge-ai==0.0.10

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 151.8/151.8 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 437.4/437.4 kB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 575.8/575.8 kB 31.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.3/419.3 kB 24.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 62.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.6/117.6 MB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 59.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 101.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.3/17.3 MB 76.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.4/56.4 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.3/253.3 kB 22.3 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found exi

You must restart the session after running the above

In [2]:

import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cuda":
    print(f"  GPU: {torch.cuda.get_device_name(0)}")


Using device: cuda
  GPU: Tesla T4


## 2 — Download Dataset

`bdanko/sjsu-headcount-scene-1` is a YOLO-format dataset (standard `data.yaml` + `train/images/` + `valid/images/` layout).
It is cloned via `git` the same way all other FOMO training datasets are fetched (marbles, RF5, etc.).

In [3]:

from pathlib import Path
import yaml
from huggingface_hub import snapshot_download

DATASET_ROOT = Path("sjsu-headcount-scene-1")
HF_REPO = "bdanko/sjsu-headcount-scene-1"

if not DATASET_ROOT.exists() or not (DATASET_ROOT / "data.yaml").exists():
    print(f"Downloading dataset {HF_REPO} via HF Hub...")
    snapshot_download(
        repo_id=HF_REPO,
        repo_type="dataset",
        local_dir=str(DATASET_ROOT),
        local_dir_use_symlinks=False
    )
    print("Dataset downloaded")
else:
    print("Dataset already present — skipping download")

# Patch data.yaml: replace relative path '.' with the absolute cloned directory
data_yaml_path = DATASET_ROOT / "data.yaml"
data_cfg = yaml.safe_load(data_yaml_path.read_text())
data_cfg["path"] = str(DATASET_ROOT.resolve())
data_yaml_path.write_text(yaml.dump(data_cfg, default_flow_style=False))

print(f"\nDataset config:")
print(f"  path : {data_cfg['path']}")
print(f"  train: {data_cfg['train']}")
print(f"  val  : {data_cfg.get('val', data_cfg.get('valid', 'N/A'))}")
print(f"  nc   : {data_cfg['nc']} ({data_cfg['names']})")

# Quick sanity check — count images and labels
def count_split(split_dir):
    imgs = list(split_dir.rglob("*.jpg")) + list(split_dir.rglob("*.png"))
    # Try split_dir.parent / "labels" first (e.g., train/labels)
    lbl_dir = split_dir.parent / "labels"
    if not lbl_dir.exists():
        # Fallback to standard Roboflow layout (labels/train)
        lbl_dir = split_dir.parent.parent / "labels" / split_dir.name
    lbls = list(lbl_dir.rglob("*.txt")) if lbl_dir.exists() else []
    return len(imgs), len(lbls)

train_imgs, train_lbls = count_split(DATASET_ROOT / "train" / "images")
val_imgs, val_lbls     = count_split(DATASET_ROOT / "valid" / "images")
print(f"Train : {train_imgs} images, {train_lbls} label files")
print(f"Val   : {val_imgs} images, {val_lbls} label files")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `snapshot_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


Fetching ... files: 0it [00:00, ?it/s]

Dataset downloaded

Dataset config:
  path : /content/sjsu-headcount-scene-1
  train: train/images
  val  : valid/images
  nc   : 1 (['person'])
Train : 406 images, 406 label files
Val   : 116 images, 116 label files


## 3 — Train FOMO-s

FOMO-m uses a **MobileNetV2 backbone (α=0.5)** with a single-pixel detection head.
Input resolution is **192** → output grid **24x24** (8× downsample).

Training uses:
- Adam, lr=3e-4, ReduceLROnPlateau on val F1
- Weighted CrossEntropy (background=1×, foreground=100×)
- No augmentation (plain resize + ImageNet normalisation)
- Val sweep over confidence thresholds every epoch

In [ ]:

from fomo import FOMO

EPOCHS     = 10
BATCH      = 16
MODEL_SIZE = "m"   # "s" | "m" | "l"
PROJECT    = "runs/fomo"
RUN_NAME   = f"sjsu_headcount_{MODEL_SIZE}"

model = FOMO(model_path=None, size=MODEL_SIZE, nb_classes=1, device=device)
print(f"Model: FOMO-{MODEL_SIZE}")
print(f"  Input size : {model.input_size}×{model.input_size}")
print(f"  Grid size  : {model.input_size//8}×{model.input_size//8}")
print(f"  Classes    : {model.nb_classes}")

import logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(message)s",
    datefmt="%H:%M:%S",
    force=True,
)

# cosine decay by default
results = model.train(
    allow_experimental=True,
    data=str(data_yaml_path),
    epochs=EPOCHS,
    batch=BATCH,
    lr0=3e-4,
    eval_interval=1,
    workers=2,
    device=device,
    project=PROJECT,
    name=RUN_NAME,
    exist_ok=True,
    patience=0,
)

save_dir = Path(results["save_dir"])
print(f"\nTraining complete. Saved to: {save_dir}")


21:58:30  Using device: cuda
21:58:30  Setting up training...
21:58:30  FOMOLoss: nc=1, fg_weight=100.0
21:58:30  FOMOLoss rebuilt with resolved dataset nc=1
21:58:30  FOMO training dataset: 406 images
21:58:30  Grid size: 24×24 (imgsz=192, downsample=8)
21:58:30  Iterations per epoch: 26 (batch_per_rank=16, world_size=1)
21:58:30  Optimizer: adam
21:58:30    - pg0 (BN): 19 params
21:58:30    - pg1 (Conv, wd=0.0): 20 params
21:58:30    - pg2 (Bias): 20 params
21:58:30  Saving to: runs/fomo/sjsu_headcount_m
21:58:30  Starting training for 10 epochs
21:58:30  Model: FOMO-m
21:58:30  Batch size: 16
21:58:30  Learning rate: 0.0003


Model: FOMO-m
  Input size : 192×192
  Grid size  : 24×24
  Classes    : 1


21:58:34  Epoch 1 - Average loss: 0.5625
21:58:34  Running FOMO validation for epoch 1
21:58:39  Epoch 1 val | loss=0.4832 | F1=0.1724 | P=0.1194 | R=0.3103 | MeanDist=0.620 | thresh=0.35 | nms_r=2 | TP=45 FP=332 FN=100 | LR=0.000293
21:58:39  New best model saved - Epoch 1: metrics/F1=0.1724
21:58:39  Checkpoint saved: runs/fomo/sjsu_headcount_m/weights/last.pt
21:58:41  Epoch 2 - Average loss: 0.3678
21:58:41  Running FOMO validation for epoch 2
21:58:44  Epoch 2 val | loss=0.3071 | F1=0.5039 | P=0.4068 | R=0.6621 | MeanDist=0.528 | thresh=0.65 | nms_r=2 | TP=96 FP=140 FN=49 | LR=0.000273
21:58:44  New best model saved - Epoch 2: metrics/F1=0.5039
21:58:44  Checkpoint saved: runs/fomo/sjsu_headcount_m/weights/last.pt
21:58:46  Epoch 3 - Average loss: 0.2586
21:58:46  Running FOMO validation for epoch 3
21:58:48  Epoch 3 val | loss=0.2227 | F1=0.6356 | P=0.5273 | R=0.8000 | MeanDist=0.413 | thresh=0.80 | nms_r=1 | TP=116 FP=104 FN=29 | LR=0.000241
21:58:48  New best model saved - Epoc


----------------------------------------------------------------
FOMO weights are hosted externally at huggingface.co/fomo-edge-ai/FOMO
under cc-by-nc-4.0.
They are treated as externally hosted, ImageNet-derived weights and
are not redistributed by FOMO. By downloading them you accept
the Hugging Face repository license terms.
----------------------------------------------------------------


Training complete. Saved to: runs/fomo/sjsu_headcount_m


In [5]:

from fomo import FOMO

weights_dir = save_dir / "weights"
best_pt  = weights_dir / "best.pt"
last_pt  = weights_dir / "last.pt"
ckpt_path = best_pt if best_pt.exists() else last_pt

trained = FOMO(str(ckpt_path), device=device)
print(f"Loaded: {ckpt_path.name}")
print(f"  family : {trained.family}")
print(f"  size   : {trained.size}")
print(f"  imgsz  : {trained.input_size}")
print(f"  nc     : {trained.nb_classes}")


Loaded: best.pt
  family : fomo
  size   : m
  imgsz  : 192
  nc     : 1


## 6 — Export to TFLite (FP32)

We can now export the trained PyTorch model to a TFLite FP32 flatbuffer using the `export` API.

In [6]:

fp32_path = trained.export(output_path=str(weights_dir / f"{RUN_NAME}_fp32.tflite"))
print(f"FP32 TFLite model exported to: {fp32_path}")


21:59:23  Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
21:59:23  Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so
21:59:26  Exporting to TFLite FP32: runs/fomo/sjsu_headcount_m/weights/sjsu_headcount_m_fp32.tflite (input 192x192, batch=1)


(00:00) [START] LiteRT-Torch Convert

(00:00) [START] LiteRT-Torch Convert > Torch Export: serving_default

(00:02) [START] LiteRT-Torch Convert > Torch Export: serving_default > ExportedProgram Run Decompositions

/usr/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


...

(00:11) [ DONE] LiteRT-Torch Convert (+00:11)

(00:00) [START] Write Model to runs/fomo/sjsu_headcount_m/weights/sjsu_headcount_m_fp32.tflite

(00:00) [ DONE] Write Model to runs/fomo/sjsu_headcount_m/weights/sjsu_headcount_m_fp32.tflite (+00:00)

21:59:37  TFLite FP32 export complete: runs/fomo/sjsu_headcount_m/weights/sjsu_headcount_m_fp32.tflite


FP32 TFLite model exported to: /content/runs/fomo/sjsu_headcount_m/weights/sjsu_headcount_m_fp32.tflite


## 7 — INT8 Quantization

To deploy the model on edge devices (like the Arm Ethos-U NPU), we quantize the weights and activations to INT8.
This uses `INT8Quantizer` with a representative calibration dataset from the validation set.

In [7]:

import numpy as np
from PIL import Image
from torchvision import transforms
from fomo.export import INT8Quantizer, INT8QuantizeConfig

val_img_dir = DATASET_ROOT / "valid" / "images"
calibration_images = sorted(val_img_dir.rglob("*.jpg"))

image_transform = transforms.Compose([
    transforms.Resize((trained.input_size, trained.input_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
])

def get_calibration_data(image_paths, num_samples=150):
    """Yields representative data formatted for the LiteRT graph."""
    count = 0
    for img_path in image_paths:
        if count >= num_samples:
            return
        img = Image.open(img_path).convert("RGB")
        tensor = image_transform(img).unsqueeze(0).numpy()
        yield {"args_0": tensor}
        count += 1

num_samples = min(len(calibration_images), 150)
print(f"Using {num_samples} validation images for calibration.")
calib_iter = get_calibration_data(calibration_images, num_samples=num_samples)

config = INT8QuantizeConfig(num_calibration_samples=num_samples)
quantizer = INT8Quantizer()

int8_path = quantizer.quantize(
    fp32_tflite=fp32_path,
    calibration_data=calib_iter,
    config=config,
    output_path=str(weights_dir / f"{RUN_NAME}_int8.tflite")
)
print(f"INT8 quantized model exported to: {int8_path}")


22:00:08  Quantizing /content/runs/fomo/sjsu_headcount_m/weights/sjsu_headcount_m_fp32.tflite → runs/fomo/sjsu_headcount_m/weights/sjsu_headcount_m_int8.tflite (samples=116)


Using 116 validation images for calibration.


22:00:09  Calibrating with 116 samples…
/usr/local/lib/python3.12/dist-packages/ai_edge_litert/interpreter.py:472: UserWarning: Warning: Enabling `experimental_preserve_all_tensors` with the BUILTIN or AUTO op resolver is intended for debugging purposes only. Be aware that this can significantly increase memory usage by storing all intermediate tensors. If you encounter memory problems or are not actively debugging, consider disabling this option.
  warnings.warn(
22:00:12  Applying quantization…
22:00:12  Serializing model...
22:00:12  INT8 TFLite export complete: runs/fomo/sjsu_headcount_m/weights/sjsu_headcount_m_int8.tflite


Model name: /content/runs/fomo/sjsu_headcount_m/weights/sjsu_headcount_m_fp32.tflite
Original model size: 130.66 KiB
Quantized model size: 99.76 KiB
Quantization Ratio: 0.76 (1.3x smaller)
Total time: 24.87 ms
INT8 quantized model exported to: /content/runs/fomo/sjsu_headcount_m/weights/sjsu_headcount_m_int8.tflite


## Vela Compilation

In [9]:
!pip install -q ethos-u-vela

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 23.9 MB/s eta 0:00:00


In [10]:
!wget https://raw.githubusercontent.com/bencejdanko/Overhead-People-Counting-YOLOXNano-FOMO-Ethos-U55-NPU/refs/heads/main/configs/default_vela.ini

!vela /content/runs/fomo/sjsu_headcount_m/weights/sjsu_headcount_m_int8.tflite \
  --accelerator-config ethos-u55-256 \
  --config /content/default_vela.ini \
  --memory-mode Shared_Sram \
  --system-config Ethos_U55_High_End_Embedded \
  --output-dir vela_output \
  --optimise Performance

--2026-06-03 22:01:49--  https://raw.githubusercontent.com/bencejdanko/Overhead-People-Counting-YOLOXNano-FOMO-Ethos-U55-NPU/refs/heads/main/configs/default_vela.ini
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.110.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 415 [text/plain]
Saving to: ‘default_vela.ini.1’

default_vela.ini.1  100%[===================>]     415  --.-KB/s    in 0s      

2026-06-03 22:01:49 (24.0 MB/s) - ‘default_vela.ini.1’ saved [415/415]


Network summary for sjsu_headcount_m_int8
Accelerator configuration               Ethos_U55_256
System configuration             Ethos_U55_High_End_Embedded
Memory mode                               Shared_Sram
Accelerator clock                                 200 MHz
Design peak SRAM bandwidth                       1.49 GB/s
Design peak Off-chip F